# 02 Pipeline

Run the canonical multi-source Orders ETL with one reusable source-block pattern and one governed target.

## Tested with FabricOps

The previous baseline was run in Microsoft Fabric with FabricOps v0.2.0 by Voyce on 6 Aug 2026. This redesigned workflow has local structural and public-API compatibility validation only; run it in your configured Fabric workspace before treating it as runtime-validated.

# 0. Environment

Load shared configuration, public APIs, one Catalogue widget, and the minimal state that must survive between cloneable blocks.

In [ ]:
%run 00_env_config

In [ ]:
from pyspark.sql import functions as F

from fabricops_kit import (
    check_dq,
    check_freshness,
    check_schema,
    profile_and_register_table,
    profile_dataframe,
    read_lakehouse_table,
    read_pipeline_prep,
    read_warehouse_query,
    write_lakehouse_table,
    write_pipeline_prep,
    widget_select_data_contract,
    widget_view_catalogue,
)

SOURCE_PREPS = {}
SOURCE_DFS = {}
PIPELINE_SHOULD_RUN = True

## Shared notebook controls

Render the Catalogue once and reuse it throughout the notebook. Development Data Contract selection is also notebook-scoped and should be rendered once when validating frozen contracts in Step 4.

In [ ]:
catalogue_widget = widget_view_catalogue(
    mode="explore",
    spark_session=spark,
)

# Step 2: leave this False while creating the initial profile/catalogue/lineage.
# Step 4: set True after Governance has frozen contracts for this notebook's lineage tables.
VALIDATE_DATA_CONTRACTS = False
CONTRACT_SELECTION = (
    widget_select_data_contract(spark_session=spark)
    if VALIDATE_DATA_CONTRACTS
    else None
)

## Pipeline configuration

Incremental source preparation needs the curated target `table_id` to read successful target-backed progress. Capture only that stable identity here; the actual TARGET Guard → Prepare → Write workflow remains under **L. Load**.

In [ ]:
# Select the registered curated Orders target in the shared Catalogue widget once.
target_anchor = catalogue_widget["get_selection"]()
if not target_anchor.get("table_id"):
    raise ValueError("Select the registered curated Orders target in the shared Catalogue widget.")

TARGET_TABLE_ID = target_anchor["table_id"]

if (
    target_anchor["store_type"] != "lakehouse"
    or target_anchor["layer"] != "unified"
    or target_anchor.get("schema_name") != "demo"
    or target_anchor["table_name"] != "orders"
):
    raise ValueError("Select the registered unified Lakehouse table demo.orders.")

print(f"Pipeline target table_id: {TARGET_TABLE_ID}")

# E. Extract

Each SOURCE block uses the same variable names and stores only the prepared context and resulting DataFrame by numeric source index. Change the small configuration cell, then reuse the same block shape.

## SOURCE 1 — Orders

Select `source.demo.orders` in the shared Catalogue widget, then run this block.

In [ ]:
SOURCE = 1
SOURCE_NAME = "Orders"
SOURCE_STORE_TYPE = "lakehouse"
SOURCE_TARGET = "source"
SOURCE_SCHEMA = "demo"
SOURCE_TABLE = "orders"
SOURCE_READER = "lakehouse_table"
SOURCE_QUERY = None
SOURCE_READ_STRATEGY = "incremental_watermark"
SOURCE_WATERMARK_COLUMN = "modified_datetime"
SOURCE_PARTITION_COLUMN = None
SOURCE_DRIVES_PIPELINE = True

In [ ]:
source_selection = catalogue_widget["get_selection"]()
if not source_selection.get("table_id"):
    raise ValueError(f"Select SOURCE {SOURCE}: {SOURCE_NAME} in the shared Catalogue widget.")

if (
    source_selection["store_type"] != SOURCE_STORE_TYPE
    or source_selection["layer"] != SOURCE_TARGET
    or source_selection.get("schema_name") != SOURCE_SCHEMA
    or source_selection["table_name"] != SOURCE_TABLE
):
    raise ValueError(f"Select the registered {SOURCE_TARGET} {SOURCE_STORE_TYPE} table {SOURCE_SCHEMA}.{SOURCE_TABLE}.")

SOURCE_TABLE_ID = source_selection["table_id"]

source_prep = read_pipeline_prep(
    source_table_id=SOURCE_TABLE_ID,
    source_read_strategy=SOURCE_READ_STRATEGY,
    target_table_id=TARGET_TABLE_ID if SOURCE_READ_STRATEGY != "full_dataset" else None,
    source_watermark_column=SOURCE_WATERMARK_COLUMN,
    source_partition_column=SOURCE_PARTITION_COLUMN,
)
SOURCE_PREPS[SOURCE] = source_prep

source_results = [check_schema(table_id=SOURCE_TABLE_ID)]
if source_prep["observation"] is not None:
    source_results.append(check_freshness(source_prep["observation"], table_id=SOURCE_TABLE_ID))
if source_prep["changes"] is not None:
    source_results.append(source_prep["changes"])
if not all(result["can_continue"] for result in source_results):
    raise RuntimeError(f"A SOURCE {SOURCE} Guardrail blocked this run.")

if SOURCE_DRIVES_PIPELINE:
    PIPELINE_SHOULD_RUN = source_prep["read_mode"] != "skip"

if PIPELINE_SHOULD_RUN:
    if SOURCE_READER == "lakehouse_table":
        source_df = read_lakehouse_table(
            SOURCE_TABLE,
            target=SOURCE_TARGET,
            schema=SOURCE_SCHEMA,
            spark_session=spark,
            processing_scope=source_prep["scope"],
        )
    elif SOURCE_READER == "warehouse_query":
        source_df = read_warehouse_query(
            SOURCE_QUERY,
            target=SOURCE_TARGET,
            spark_session=spark,
        )
    else:
        raise ValueError(f"Unsupported SOURCE_READER: {SOURCE_READER}")

    source_dq = check_dq(source_df, table_id=SOURCE_TABLE_ID)
    display(source_dq["summary"])
    if not source_dq["can_continue"]:
        raise RuntimeError(f"A SOURCE {SOURCE} DQ Guardrail blocked this run.")

    if SOURCE_READER == "warehouse_query":
        source_profile = profile_dataframe(source_df)
    elif source_prep["read_mode"] == "full_dataset":
        source_profile = profile_and_register_table(
            source_df,
            profile_role="source",
            table=source_prep["source"],
        )
    else:
        source_profile = profile_dataframe(source_df)

    SOURCE_DFS[SOURCE] = source_df
    display(source_profile)
else:
    SOURCE_DFS[SOURCE] = None
    print(f"SOURCE {SOURCE} has no work. Downstream physical processing is skipped.")

## SOURCE 2 — Products

Select `source.demo.products` in the shared Catalogue widget, then run the same SOURCE pattern.

In [ ]:
SOURCE = 2
SOURCE_NAME = "Products"
SOURCE_STORE_TYPE = "lakehouse"
SOURCE_TARGET = "source"
SOURCE_SCHEMA = "demo"
SOURCE_TABLE = "products"
SOURCE_READER = "lakehouse_table"
SOURCE_QUERY = None
SOURCE_READ_STRATEGY = "full_dataset"
SOURCE_WATERMARK_COLUMN = None
SOURCE_PARTITION_COLUMN = None
SOURCE_DRIVES_PIPELINE = False

In [ ]:
source_selection = catalogue_widget["get_selection"]()
if not source_selection.get("table_id"):
    raise ValueError(f"Select SOURCE {SOURCE}: {SOURCE_NAME} in the shared Catalogue widget.")

if (
    source_selection["store_type"] != SOURCE_STORE_TYPE
    or source_selection["layer"] != SOURCE_TARGET
    or source_selection.get("schema_name") != SOURCE_SCHEMA
    or source_selection["table_name"] != SOURCE_TABLE
):
    raise ValueError(f"Select the registered {SOURCE_TARGET} {SOURCE_STORE_TYPE} table {SOURCE_SCHEMA}.{SOURCE_TABLE}.")

SOURCE_TABLE_ID = source_selection["table_id"]

source_prep = read_pipeline_prep(
    source_table_id=SOURCE_TABLE_ID,
    source_read_strategy=SOURCE_READ_STRATEGY,
    target_table_id=TARGET_TABLE_ID if SOURCE_READ_STRATEGY != "full_dataset" else None,
    source_watermark_column=SOURCE_WATERMARK_COLUMN,
    source_partition_column=SOURCE_PARTITION_COLUMN,
)
SOURCE_PREPS[SOURCE] = source_prep

source_results = [check_schema(table_id=SOURCE_TABLE_ID)]
if source_prep["observation"] is not None:
    source_results.append(check_freshness(source_prep["observation"], table_id=SOURCE_TABLE_ID))
if source_prep["changes"] is not None:
    source_results.append(source_prep["changes"])
if not all(result["can_continue"] for result in source_results):
    raise RuntimeError(f"A SOURCE {SOURCE} Guardrail blocked this run.")

if SOURCE_DRIVES_PIPELINE:
    PIPELINE_SHOULD_RUN = source_prep["read_mode"] != "skip"

if PIPELINE_SHOULD_RUN:
    if SOURCE_READER == "lakehouse_table":
        source_df = read_lakehouse_table(
            SOURCE_TABLE,
            target=SOURCE_TARGET,
            schema=SOURCE_SCHEMA,
            spark_session=spark,
            processing_scope=source_prep["scope"],
        )
    elif SOURCE_READER == "warehouse_query":
        source_df = read_warehouse_query(
            SOURCE_QUERY,
            target=SOURCE_TARGET,
            spark_session=spark,
        )
    else:
        raise ValueError(f"Unsupported SOURCE_READER: {SOURCE_READER}")

    source_dq = check_dq(source_df, table_id=SOURCE_TABLE_ID)
    display(source_dq["summary"])
    if not source_dq["can_continue"]:
        raise RuntimeError(f"A SOURCE {SOURCE} DQ Guardrail blocked this run.")

    if SOURCE_READER == "warehouse_query":
        source_profile = profile_dataframe(source_df)
    elif source_prep["read_mode"] == "full_dataset":
        source_profile = profile_and_register_table(
            source_df,
            profile_role="source",
            table=source_prep["source"],
        )
    else:
        source_profile = profile_dataframe(source_df)

    SOURCE_DFS[SOURCE] = source_df
    display(source_profile)
else:
    SOURCE_DFS[SOURCE] = None
    print(f"SOURCE {SOURCE} has no work. Downstream physical processing is skipped.")

## SOURCE 3 — Order History

Select `product.demo.order_history` in the shared Catalogue widget. Only the configuration changes; the runner remains the same pattern.

In [ ]:
SOURCE = 3
SOURCE_NAME = "Order History"
SOURCE_STORE_TYPE = "warehouse"
SOURCE_TARGET = "product"
SOURCE_SCHEMA = "demo"
SOURCE_TABLE = "order_history"
SOURCE_READER = "warehouse_query"
SOURCE_QUERY = """
SELECT
    customer_id,
    COUNT(*) AS historical_order_count,
    SUM(net_amount) AS historical_net_amount,
    MAX(order_datetime) AS latest_historical_order_datetime
FROM demo.order_history
GROUP BY customer_id
"""
SOURCE_READ_STRATEGY = "full_dataset"
SOURCE_WATERMARK_COLUMN = None
SOURCE_PARTITION_COLUMN = None
SOURCE_DRIVES_PIPELINE = False

In [ ]:
source_selection = catalogue_widget["get_selection"]()
if not source_selection.get("table_id"):
    raise ValueError(f"Select SOURCE {SOURCE}: {SOURCE_NAME} in the shared Catalogue widget.")

if (
    source_selection["store_type"] != SOURCE_STORE_TYPE
    or source_selection["layer"] != SOURCE_TARGET
    or source_selection.get("schema_name") != SOURCE_SCHEMA
    or source_selection["table_name"] != SOURCE_TABLE
):
    raise ValueError(f"Select the registered {SOURCE_TARGET} {SOURCE_STORE_TYPE} table {SOURCE_SCHEMA}.{SOURCE_TABLE}.")

SOURCE_TABLE_ID = source_selection["table_id"]

source_prep = read_pipeline_prep(
    source_table_id=SOURCE_TABLE_ID,
    source_read_strategy=SOURCE_READ_STRATEGY,
    target_table_id=TARGET_TABLE_ID if SOURCE_READ_STRATEGY != "full_dataset" else None,
    source_watermark_column=SOURCE_WATERMARK_COLUMN,
    source_partition_column=SOURCE_PARTITION_COLUMN,
)
SOURCE_PREPS[SOURCE] = source_prep

source_results = [check_schema(table_id=SOURCE_TABLE_ID)]
if source_prep["observation"] is not None:
    source_results.append(check_freshness(source_prep["observation"], table_id=SOURCE_TABLE_ID))
if source_prep["changes"] is not None:
    source_results.append(source_prep["changes"])
if not all(result["can_continue"] for result in source_results):
    raise RuntimeError(f"A SOURCE {SOURCE} Guardrail blocked this run.")

if SOURCE_DRIVES_PIPELINE:
    PIPELINE_SHOULD_RUN = source_prep["read_mode"] != "skip"

if PIPELINE_SHOULD_RUN:
    if SOURCE_READER == "lakehouse_table":
        source_df = read_lakehouse_table(
            SOURCE_TABLE,
            target=SOURCE_TARGET,
            schema=SOURCE_SCHEMA,
            spark_session=spark,
            processing_scope=source_prep["scope"],
        )
    elif SOURCE_READER == "warehouse_query":
        source_df = read_warehouse_query(
            SOURCE_QUERY,
            target=SOURCE_TARGET,
            spark_session=spark,
        )
    else:
        raise ValueError(f"Unsupported SOURCE_READER: {SOURCE_READER}")

    source_dq = check_dq(source_df, table_id=SOURCE_TABLE_ID)
    display(source_dq["summary"])
    if not source_dq["can_continue"]:
        raise RuntimeError(f"A SOURCE {SOURCE} DQ Guardrail blocked this run.")

    if SOURCE_READER == "warehouse_query":
        source_profile = profile_dataframe(source_df)
    elif source_prep["read_mode"] == "full_dataset":
        source_profile = profile_and_register_table(
            source_df,
            profile_role="source",
            table=source_prep["source"],
        )
    else:
        source_profile = profile_dataframe(source_df)

    SOURCE_DFS[SOURCE] = source_df
    display(source_profile)
else:
    SOURCE_DFS[SOURCE] = None
    print(f"SOURCE {SOURCE} has no work. Downstream physical processing is skipped.")

### Strategy and runtime scope

Configured source strategies are `full_dataset`, `incremental_watermark`, or `incremental_partition`. Preparation separately returns runtime modes `full_dataset`, `incremental_subset`, or `skip`. Only a complete table read replaces the canonical source profile; incremental slices and query aggregates are diagnostic.

# T. Transform

**Business transformation is project-owned.** Use the three stored source DataFrames; FabricOps does not hide the business joins.

In [ ]:
if PIPELINE_SHOULD_RUN:
    orders_df = SOURCE_DFS[1].alias("orders")
    products_df = SOURCE_DFS[2].alias("products")
    history_df = SOURCE_DFS[3].alias("history")

    transformed_df = (
        orders_df
        .join(products_df, on="product_id", how="left")
        .join(history_df, on="customer_id", how="left")
        .withColumn(
            "order_net_amount",
            F.round(F.col("quantity") * F.col("unit_price") * (F.lit(1.0) - F.col("discount")), 2),
        )
        .fillna({"historical_order_count": 0, "historical_net_amount": 0.0})
        .select(
            "order_id", "customer_id", "order_datetime", "modified_datetime",
            "product_id", "product_name", "product_category", "quantity", "unit_price",
            "discount", "order_net_amount", "order_status", "shipping_country",
            "historical_order_count", "historical_net_amount", "latest_historical_order_datetime",
        )
    )
    display(transformed_df)

# L. Load

## TARGET 1 — Curated Orders

Select `unified.demo.orders` in the same shared Catalogue widget. The target workflow starts here: Select → Guard → Prepare → Write → Evidence.

In [ ]:
TARGET = 1
TARGET_NAME = "Curated Orders"
TARGET_STORE_TYPE = "lakehouse"
TARGET_TARGET = "unified"
TARGET_SCHEMA = "demo"
TARGET_TABLE = "orders"

target_selection = catalogue_widget["get_selection"]()
if not target_selection.get("table_id"):
    raise ValueError(f"Select TARGET {TARGET}: {TARGET_NAME} in the shared Catalogue widget.")

if (
    target_selection["store_type"] != TARGET_STORE_TYPE
    or target_selection["layer"] != TARGET_TARGET
    or target_selection.get("schema_name") != TARGET_SCHEMA
    or target_selection["table_name"] != TARGET_TABLE
    or target_selection["table_id"] != TARGET_TABLE_ID
):
    raise ValueError(f"Select the registered {TARGET_TARGET} {TARGET_STORE_TYPE} table {TARGET_SCHEMA}.{TARGET_TABLE}.")

target_table_id = target_selection["table_id"]
print(f"TARGET {TARGET} table_id: {target_table_id}")

In [ ]:
if PIPELINE_SHOULD_RUN:
    target_df = transformed_df
    target_schema_result = check_schema(table_id=target_table_id, dataframe=target_df)
    target_dq_result = check_dq(target_df, table_id=target_table_id)
    display(target_dq_result["summary"])

    if not all(result["can_continue"] for result in (target_schema_result, target_dq_result)):
        raise RuntimeError(f"A TARGET {TARGET} Guardrail blocked publication.")

In [ ]:
if PIPELINE_SHOULD_RUN:
    target_prep = write_pipeline_prep(
        target_df,
        target_table_id=target_table_id,
        source_preps=[SOURCE_PREPS[1], SOURCE_PREPS[2], SOURCE_PREPS[3]],
    )
    prepared_target_df = target_prep["df"].persist()

### Distributed write processing / Publish

`repartition_by=4` explicitly redistributes the prepared DataFrame so Spark can process the write across available executors. It does not create Python threads or multiple independent writers.

In [ ]:
if PIPELINE_SHOULD_RUN:
    target_slice_profile = profile_dataframe(prepared_target_df)

    write_lakehouse_table(
        prepared_target_df,
        target_prep["target"]["table_name"],
        target=target_prep["target"]["layer"],
        schema=target_prep["target"].get("schema_name"),
        mode=target_prep["mode"],
        options=target_prep["options"],
        load_strategy=target_prep["load_strategy"],
        load_strategy_parameters=target_prep["load_strategy_parameters"],
        processing_scope=target_prep["scope"],
        repartition_by=4,
    )
    prepared_target_df.unpersist()

### Evidence

After successful publication, profile the complete curated table. This writes coherent `METADATA_DATA_PROFILED`, eligible `METADATA_DATA_PROFILED_FREQUENCY`, and `METADATA_DATA_CATALOGUE` records; preparation/publication retain `METADATA_DATA_LINEAGE` participation.

In [ ]:
if PIPELINE_SHOULD_RUN:
    curated_orders_df = read_lakehouse_table(
        target_prep["target"]["table_name"],
        target=target_prep["target"]["layer"],
        schema=target_prep["target"].get("schema_name"),
        spark_session=spark,
    )
    target_profile = profile_and_register_table(
        curated_orders_df,
        profile_role="target",
        table=target_prep["target"],
        load_strategy=target_prep["load_strategy"],
        load_strategy_parameters=target_prep["load_strategy_parameters"],
    )
    display(target_profile)
    print(f"Curated Orders table_id: {target_table_id}")